# Phase 1: Taste Embedding Model - Example Usage

This notebook demonstrates how to use the trained product embedding model for similarity search and clustering.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
from evaluate import EmbeddingEvaluator
import json


## Load Embeddings and Metadata


In [ ]:
# Initialize evaluator
embeddings_path = Path('../embeddings/product_embeddings.npy')
metadata_path = Path('../embeddings/product_metadata.json')
faiss_index_path = Path('../embeddings/faiss_index.bin')

evaluator = EmbeddingEvaluator(
    embeddings_path,
    metadata_path,
    faiss_index_path
)


## Similarity Search Examples


In [ ]:
# Find products similar to a specific product
product_name = "Coca Cola- 16 fl oz"  # Change this to any product name

similar_products = evaluator.find_similar_products(
    product_name=product_name,
    top_k=10
)

print(f"Products similar to '{product_name}':\n")
for i, (pid, name, similarity) in enumerate(similar_products, 1):
    print(f"{i}. {name} (similarity: {similarity:.4f})")


## Clustering Analysis


In [ ]:
# Cluster products
cluster_results = evaluator.cluster_products(n_clusters=10)

# Display cluster information
print(f"Found {cluster_results['n_clusters']} clusters\n")
for cluster_id, products in cluster_results['clusters'].items():
    print(f"Cluster {cluster_id} ({len(products)} products):")
    for product in products[:5]:  # Show first 5 products
        print(f"  - {product['product_name']}")
    if len(products) > 5:
        print(f"  ... and {len(products) - 5} more")
    print()


## Category Separation Analysis


In [ ]:
# Load products dataframe
products_df = pd.read_csv('../data/processed/products.csv')

# Evaluate category separation
separation_results = evaluator.evaluate_category_separation(products_df)

print("Category Separation Metrics:")
print(f"Intra-category similarity (mean): {separation_results['intra_category_mean']:.4f}")
print(f"Inter-category similarity (mean): {separation_results['inter_category_mean']:.4f}")
print(f"Separation score: {separation_results['separation_score']:.4f}")
print("\n(Higher separation score = better category separation)")


## Sanity Checks


In [ ]:
# Perform sanity checks
sanity_results = evaluator.sanity_check(products_df)

print("Sanity Check Results:\n")
for test_name, results in sanity_results.items():
    print(f"{test_name}:")
    print(f"  Within group 1 similarity: {results['within_group1']:.4f}")
    print(f"  Within group 2 similarity: {results['within_group2']:.4f}")
    print(f"  Between groups similarity: {results['between_groups']:.4f}")
    print()


## Visualization



In [ ]:
# Visualize clusters
cluster_labels = np.array(cluster_results['cluster_labels'])
evaluator.visualize_clusters(cluster_labels, n_samples=200)
